In [56]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix


In [57]:
# ===============================================================
# 1. Läs in data
# ===============================================================
df = pd.read_csv("Pokemon.csv")

In [58]:
# Behåll bara features + target
df = df.drop(columns=["Type 2", "Legendary"], errors="ignore")

X = df[['HP','Attack','Defense','Sp. Atk','Sp. Def','Speed','Generation']]
y = df['Type 1']

# Alla klasser i korrekt ordning (fixar problem med olika CM-storlekar)
labels = sorted(y.unique())

In [68]:
# ===============================================================
# 2. Funktion som kör ett experiment 100 gånger
# ===============================================================
def run_experiment(X_data, y_data, train_size, use_normalization=False, description=""):
    accuracies = []
    best_acc = -1
    best_cm = None
    
    print(f"\n{'='*60}")
    print(f"EXPERIMENT: {description}")
    print(f"Tränings-/Test-förhållande: {train_size:.0%}/{1-train_size:.0%}")
    print(f"Normalisering: {'JA' if use_normalization else 'NEJ'}")
    print(f"{'='*60}")
    
    for i in range(100):
        # Split data med random_state=None som specificerat
        X_train, X_test, y_train, y_test = train_test_split(
            X_data, y_data,
            train_size=train_size,
            random_state=None,  # <-- HÄR är random_state=None
            stratify=y_data
        )
        
        # Normalisering om det är aktiverat
        if use_normalization:
            scaler = MinMaxScaler(feature_range=(0, 1))
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
        
        # Träna modell med random_state=None
        clf = DecisionTreeClassifier(random_state=None)  # <-- HÄR också
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        # Beräkna accuracy
        acc = accuracy_score(y_test, y_pred)
        accuracies.append(acc)
        
        # Spara bästa resultatet
        if acc > best_acc:
            best_acc = acc
            best_cm = confusion_matrix(y_test, y_pred, labels=labels)
    
    # Beräkna statistik
    mean_acc = np.mean(accuracies)
    std_acc = np.std(accuracies)
    min_acc = np.min(accuracies)
    max_acc = np.max(accuracies)
    
    # Presentera resultat
    print(f"\nRESULTAT (100 körningar):")
    print(f"Genomsnittlig accuracy: {mean_acc:.4f}")
    print(f"Standardavvikelse: {std_acc:.4f}")
    print(f"Lägsta accuracy: {min_acc:.4f}")
    print(f"Högsta accuracy: {max_acc:.4f}")
    print(f"Bästa accuracy: {best_acc:.4f}")
    
    return mean_acc, best_acc, best_cm, accuracies

In [74]:


# ===============================================================
# 3. Kör alla experiment
# ===============================================================

# Experiment 1: Icke-normaliserat data med 90/10 split
mean_acc1, best_acc1, best_cm1, accuracies1 = run_experiment(
    X, y, 
    train_size=0.90,
    use_normalization=False,
    description="Icke-normaliserat data"
)

# Experiment 2: Normaliserat data med 90/10 split
mean_acc2, best_acc2, best_cm2, accuracies2 = run_experiment(
    X, y,
    train_size=0.90,
    use_normalization=True,
    description="Normaliserat data [0,1]"
)

# Experiment 3: Icke-normaliserat data med eget valt förhållande (70/30)
mean_acc3, best_acc3, best_cm3, accuracies3 = run_experiment(
    X, y,
    train_size=0.70,
    use_normalization=False,
    description="Icke-normaliserat data"
)

# Experiment 4: Normaliserat data med eget valt förhållande (70/30)
mean_acc4, best_acc4, best_cm4, accuracies4 = run_experiment(
    X, y,
    train_size=0.70,
    use_normalization=True,
    description="Normaliserat data [0,1]"
)


EXPERIMENT: Icke-normaliserat data
Tränings-/Test-förhållande: 90%/10%
Normalisering: NEJ

RESULTAT (100 körningar):
Genomsnittlig accuracy: 0.1778
Standardavvikelse: 0.0388
Lägsta accuracy: 0.0875
Högsta accuracy: 0.2625
Bästa accuracy: 0.2625

EXPERIMENT: Normaliserat data [0,1]
Tränings-/Test-förhållande: 90%/10%
Normalisering: JA

RESULTAT (100 körningar):
Genomsnittlig accuracy: 0.1763
Standardavvikelse: 0.0394
Lägsta accuracy: 0.0750
Högsta accuracy: 0.2875
Bästa accuracy: 0.2875

EXPERIMENT: Icke-normaliserat data
Tränings-/Test-förhållande: 70%/30%
Normalisering: NEJ

RESULTAT (100 körningar):
Genomsnittlig accuracy: 0.1670
Standardavvikelse: 0.0257
Lägsta accuracy: 0.1125
Högsta accuracy: 0.2292
Bästa accuracy: 0.2292

EXPERIMENT: Normaliserat data [0,1]
Tränings-/Test-förhållande: 70%/30%
Normalisering: JA

RESULTAT (100 körningar):
Genomsnittlig accuracy: 0.1664
Standardavvikelse: 0.0223
Lägsta accuracy: 0.1042
Högsta accuracy: 0.2333
Bästa accuracy: 0.2333


In [75]:
# ===============================================================
# 4. Jämförelse mellan normaliserat och icke-normaliserat data
# ===============================================================
print("\n" + "="*80)
print("SAMMANFATTNING - JÄMFÖRELSE MELLAN NORMALISERAT OCH ICKE-NORMALISERAT DATA")
print("="*80)

print(f"\n{'Experiment':<50} {'Medelaccuracy':<15} {'Skillnad':<10}")
print("-"*75)

print(f"{'1. Icke-normaliserat (90/10)':<50} {mean_acc1:.4f}")
print(f"{'2. Normaliserat [0,1] (90/10)':<50} {mean_acc2:.4f} {f'({(mean_acc2-mean_acc1)*100:+.2f}%)':<10}")
print(f"{'3. Icke-normaliserat (70/30)':<50} {mean_acc3:.4f}")
print(f"{'4. Normaliserat [0,1] (70/30)':<50} {mean_acc4:.4f} {f'({(mean_acc4-mean_acc3)*100:+.2f}%)':<10}")


SAMMANFATTNING - JÄMFÖRELSE MELLAN NORMALISERAT OCH ICKE-NORMALISERAT DATA

Experiment                                         Medelaccuracy   Skillnad  
---------------------------------------------------------------------------
1. Icke-normaliserat (90/10)                       0.1778
2. Normaliserat [0,1] (90/10)                      0.1763 (-0.15%)  
3. Icke-normaliserat (70/30)                       0.1670
4. Normaliserat [0,1] (70/30)                      0.1664 (-0.06%)  


In [76]:
# ===============================================================
# 5. Analys av skillnader
# ===============================================================
print("\n" + "="*80)
print("ANALYS AV RESULTAT")
print("="*80)

print(f"\n1. Påverkan av normalisering (90/10 split):")
print(f"   - Icke-normaliserat: {mean_acc1:.4f}")
print(f"   - Normaliserat: {mean_acc2:.4f}")
print(f"   - Skillnad: {(mean_acc2 - mean_acc1):.4f} ({(mean_acc2 - mean_acc1)*100:+.2f}%)")

print(f"\n2. Påverkan av normalisering (70/30 split):")
print(f"   - Icke-normaliserat: {mean_acc3:.4f}")
print(f"   - Normaliserat: {mean_acc4:.4f}")
print(f"   - Skillnad: {(mean_acc4 - mean_acc3):.4f} ({(mean_acc4 - mean_acc3)*100:+.2f}%)")

print(f"\n3. Påverkan av träningsstorlek (icke-normaliserat):")
print(f"   - 90/10 split: {mean_acc1:.4f}")
print(f"   - 70/30 split: {mean_acc3:.4f}")
print(f"   - Skillnad: {(mean_acc1 - mean_acc3):.4f} ({(mean_acc1 - mean_acc3)*100:+.2f}%)")

print(f"\n4. Påverkan av träningsstorlek (normaliserat):")
print(f"   - 90/10 split: {mean_acc2:.4f}")
print(f"   - 70/30 split: {mean_acc4:.4f}")
print(f"   - Skillnad: {(mean_acc2 - mean_acc4):.4f} ({(mean_acc2 - mean_acc4)*100:+.2f}%)")




ANALYS AV RESULTAT

1. Påverkan av normalisering (90/10 split):
   - Icke-normaliserat: 0.1778
   - Normaliserat: 0.1763
   - Skillnad: -0.0015 (-0.15%)

2. Påverkan av normalisering (70/30 split):
   - Icke-normaliserat: 0.1670
   - Normaliserat: 0.1664
   - Skillnad: -0.0006 (-0.06%)

3. Påverkan av träningsstorlek (icke-normaliserat):
   - 90/10 split: 0.1778
   - 70/30 split: 0.1670
   - Skillnad: 0.0107 (+1.07%)

4. Påverkan av träningsstorlek (normaliserat):
   - 90/10 split: 0.1763
   - 70/30 split: 0.1664
   - Skillnad: 0.0098 (+0.98%)


In [77]:
# ===============================================================
# 6. Visa confusion matrices för bästa körningarna
# ===============================================================
print("\n" + "="*80)
print("CONFUSION MATRICES FÖR BÄSTA KÖRNINGARNA")
print("="*80)

experiments = [
    ("1. Icke-normaliserat (90/10)", best_cm1, best_acc1),
    ("2. Normaliserat [0,1] (90/10)", best_cm2, best_acc2),
    ("3. Icke-normaliserat (70/30)", best_cm3, best_acc3),
    ("4. Normaliserat [0,1] (70/30)", best_cm4, best_acc4)
]

for desc, cm, acc in experiments:
    print(f"\n{desc} (Accuracy: {acc:.4f}):")
    print(f"Storlek på confusion matrix: {cm.shape}")
    print(f"Rätt klassificerade (diagonalen): {np.trace(cm)}")
    print(f"Totala testfall: {np.sum(cm)}")
    
    # Visa en sammanfattning av confusion matrix
    print("\nFörsta 5x5 delen av confusion matrix:")
    print("-" * 50)
    # Skriv ut första 5 rader och kolumner
    for i in range(min(5, cm.shape[0])):
        row = ""
        for j in range(min(5, cm.shape[1])):
            row += f"{cm[i, j]:4d}"
        if cm.shape[1] > 5:
            row += " ..."
        print(row)
    if cm.shape[0] > 5:
        print("...")



CONFUSION MATRICES FÖR BÄSTA KÖRNINGARNA

1. Icke-normaliserat (90/10) (Accuracy: 0.2625):
Storlek på confusion matrix: (18, 18)
Rätt klassificerade (diagonalen): 21
Totala testfall: 80

Första 5x5 delen av confusion matrix:
--------------------------------------------------
   2   0   0   0   0 ...
   1   0   0   0   0 ...
   0   0   1   1   0 ...
   0   0   0   2   0 ...
   1   0   0   0   0 ...
...

2. Normaliserat [0,1] (90/10) (Accuracy: 0.2875):
Storlek på confusion matrix: (18, 18)
Rätt klassificerade (diagonalen): 23
Totala testfall: 80

Första 5x5 delen av confusion matrix:
--------------------------------------------------
   3   1   0   1   0 ...
   1   0   0   0   1 ...
   0   0   1   0   0 ...
   0   0   0   2   0 ...
   0   0   0   0   0 ...
...

3. Icke-normaliserat (70/30) (Accuracy: 0.2292):
Storlek på confusion matrix: (18, 18)
Rätt klassificerade (diagonalen): 55
Totala testfall: 240

Första 5x5 delen av confusion matrix:
--------------------------------------------

In [78]:
# ===============================================================
# 7. Statistik för variation mellan körningar
# ===============================================================
print("\n" + "="*80)
print("VARIATION MELLAN KÖRNINGAR")
print("="*80)

print(f"\nStandardavvikelse (mindre = mer stabila resultat):")
print(f"1. Icke-normaliserat (90/10): {np.std(accuracies1):.4f}")
print(f"2. Normaliserat (90/10): {np.std(accuracies2):.4f}")
print(f"3. Icke-normaliserat (70/30): {np.std(accuracies3):.4f}")
print(f"4. Normaliserat (70/30): {np.std(accuracies4):.4f}")

print(f"\nRange (max - min):")
print(f"1. Icke-normaliserat (90/10): {np.max(accuracies1)-np.min(accuracies1):.4f}")
print(f"2. Normaliserat (90/10): {np.max(accuracies2)-np.min(accuracies2):.4f}")
print(f"3. Icke-normaliserat (70/30): {np.max(accuracies3)-np.min(accuracies3):.4f}")
print(f"4. Normaliserat (70/30): {np.max(accuracies4)-np.min(accuracies4):.4f}")


VARIATION MELLAN KÖRNINGAR

Standardavvikelse (mindre = mer stabila resultat):
1. Icke-normaliserat (90/10): 0.0388
2. Normaliserat (90/10): 0.0394
3. Icke-normaliserat (70/30): 0.0257
4. Normaliserat (70/30): 0.0223

Range (max - min):
1. Icke-normaliserat (90/10): 0.1750
2. Normaliserat (90/10): 0.2125
3. Icke-normaliserat (70/30): 0.1167
4. Normaliserat (70/30): 0.1292
